### Allscripts Sunrise (SCM) Drug Exposure Hydration Diagnostics

Run this notebook after rerunning SCM drug exposure when `drug_concept_id` is still zero.
It counts each join stage on the live hydration path so we can see where the concept bridge fails.

In [ ]:
%sql
SELECT COUNT(*) AS medication_order_rows
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.GenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS generic_item_matches,
  SUM(CASE WHEN gi.RxNormCode IS NOT NULL AND TRIM(CAST(gi.RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS rows_with_rxnorm
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
 AND gi.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN rxnorm_source.concept_id IS NOT NULL THEN 1 ELSE 0 END) AS rxnorm_source_matches,
  SUM(CASE WHEN rxnorm_standard.concept_id IS NOT NULL THEN 1 ELSE 0 END) AS rxnorm_standard_matches
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
 AND gi.Active = TRUE
LEFT JOIN _exponent.omop.concept rxnorm_source
  ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
 AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
 AND rxnorm_source.domain_id = 'Drug'
 AND rxnorm_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship rxnorm_maps_to
  ON rxnorm_maps_to.concept_id_1 = rxnorm_source.concept_id
 AND rxnorm_maps_to.relationship_id = 'Maps to'
 AND rxnorm_maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept rxnorm_standard
  ON rxnorm_standard.concept_id = rxnorm_maps_to.concept_id_2
 AND rxnorm_standard.standard_concept = 'S'
 AND rxnorm_standard.domain_id = 'Drug'
 AND rxnorm_standard.invalid_reason IS NULL
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN p.ProductID IS NOT NULL THEN 1 ELSE 0 END) AS product_matches,
  SUM(CASE WHEN pp.ProductPackageID IS NOT NULL THEN 1 ELSE 0 END) AS package_matches,
  SUM(CASE WHEN pp.NDCCode IS NOT NULL AND TRIM(CAST(pp.NDCCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS rows_with_ndc,
  SUM(CASE WHEN ndc_source.concept_id IS NOT NULL THEN 1 ELSE 0 END) AS ndc_source_matches,
  SUM(CASE WHEN ndc_standard.concept_id IS NOT NULL THEN 1 ELSE 0 END) AS ndc_standard_matches
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
 AND gi.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
  ON p.GenericItemID = gi.GenericItemID
 AND p.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage pp
  ON pp.ProductID = p.ProductID
 AND pp.Active = TRUE
LEFT JOIN _exponent.omop.concept ndc_source
  ON ndc_source.concept_code = REGEXP_REPLACE(TRIM(CAST(pp.NDCCode AS STRING)), '[^0-9]', '')
 AND ndc_source.vocabulary_id = 'NDC'
 AND ndc_source.domain_id = 'Drug'
 AND ndc_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship ndc_maps_to
  ON ndc_maps_to.concept_id_1 = ndc_source.concept_id
 AND ndc_maps_to.relationship_id = 'Maps to'
 AND ndc_maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept ndc_standard
  ON ndc_standard.concept_id = ndc_maps_to.concept_id_2
 AND ndc_standard.standard_concept = 'S'
 AND ndc_standard.domain_id = 'Drug'
 AND ndc_standard.invalid_reason IS NULL
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  medext.PrescriptionGenericItemID,
  ord.Name,
  medext.OrderedAs,
  medext.OrderedAsDisplay,
  medext.MultumDrugName,
  gi.GenericItemID,
  gi.DrugID,
  gi.GenericItemName,
  gi.RxNormCode,
  p.ProductID,
  p.BrandName,
  pp.ProductPackageID,
  pp.NDCCode,
  rxnorm_source.concept_id AS rxnorm_source_concept_id,
  rxnorm_standard.concept_id AS rxnorm_standard_concept_id,
  ndc_source.concept_id AS ndc_source_concept_id,
  ndc_standard.concept_id AS ndc_standard_concept_id,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
 AND gi.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
  ON p.GenericItemID = gi.GenericItemID
 AND p.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage pp
  ON pp.ProductID = p.ProductID
 AND pp.Active = TRUE
LEFT JOIN _exponent.omop.concept rxnorm_source
  ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
 AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
 AND rxnorm_source.domain_id = 'Drug'
 AND rxnorm_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship rxnorm_maps_to
  ON rxnorm_maps_to.concept_id_1 = rxnorm_source.concept_id
 AND rxnorm_maps_to.relationship_id = 'Maps to'
 AND rxnorm_maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept rxnorm_standard
  ON rxnorm_standard.concept_id = rxnorm_maps_to.concept_id_2
 AND rxnorm_standard.standard_concept = 'S'
 AND rxnorm_standard.domain_id = 'Drug'
 AND rxnorm_standard.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept ndc_source
  ON ndc_source.concept_code = REGEXP_REPLACE(TRIM(CAST(pp.NDCCode AS STRING)), '[^0-9]', '')
 AND ndc_source.vocabulary_id = 'NDC'
 AND ndc_source.domain_id = 'Drug'
 AND ndc_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship ndc_maps_to
  ON ndc_maps_to.concept_id_1 = ndc_source.concept_id
 AND ndc_maps_to.relationship_id = 'Maps to'
 AND ndc_maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept ndc_standard
  ON ndc_standard.concept_id = ndc_maps_to.concept_id_2
 AND ndc_standard.standard_concept = 'S'
 AND ndc_standard.domain_id = 'Drug'
 AND ndc_standard.invalid_reason IS NULL
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
GROUP BY medext.PrescriptionGenericItemID, ord.Name, medext.OrderedAs, medext.OrderedAsDisplay, medext.MultumDrugName, gi.GenericItemID, gi.DrugID, gi.GenericItemName, gi.RxNormCode, p.ProductID, p.BrandName, pp.ProductPackageID, pp.NDCCode, rxnorm_source.concept_id, rxnorm_standard.concept_id, ndc_source.concept_id, ndc_standard.concept_id
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
WITH mapping_candidates AS (
  SELECT
    CASE
      WHEN rxnorm_standard.concept_id IS NOT NULL THEN 'rxnorm_standard'
      WHEN ndc_standard.concept_id IS NOT NULL THEN 'ndc_standard'
      WHEN rxnorm_source.concept_id IS NOT NULL THEN 'rxnorm_source'
      WHEN ndc_source.concept_id IS NOT NULL THEN 'ndc_source'
      ELSE 'unmapped'
    END AS winning_path,
    CASE
      WHEN rxnorm_standard.concept_id IS NOT NULL THEN rxnorm_standard.concept_id
      WHEN ndc_standard.concept_id IS NOT NULL THEN ndc_standard.concept_id
      WHEN rxnorm_source.concept_id IS NOT NULL THEN rxnorm_source.concept_id
      WHEN ndc_source.concept_id IS NOT NULL THEN ndc_source.concept_id
      ELSE 0
    END AS derived_drug_concept_id
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
  INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
    ON medext.GUID = ord.GUID
   AND medext.Active = TRUE
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
    ON gi.GenericItemID = medext.PrescriptionGenericItemID
   AND gi.Active = TRUE
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
    ON p.GenericItemID = gi.GenericItemID
   AND p.Active = TRUE
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage pp
    ON pp.ProductID = p.ProductID
   AND pp.Active = TRUE
  LEFT JOIN _exponent.omop.concept rxnorm_source
    ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
   AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
   AND rxnorm_source.domain_id = 'Drug'
   AND rxnorm_source.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept_relationship rxnorm_maps_to
    ON rxnorm_maps_to.concept_id_1 = rxnorm_source.concept_id
   AND rxnorm_maps_to.relationship_id = 'Maps to'
   AND rxnorm_maps_to.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept rxnorm_standard
    ON rxnorm_standard.concept_id = rxnorm_maps_to.concept_id_2
   AND rxnorm_standard.standard_concept = 'S'
   AND rxnorm_standard.domain_id = 'Drug'
   AND rxnorm_standard.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept ndc_source
    ON ndc_source.concept_code = REGEXP_REPLACE(TRIM(CAST(pp.NDCCode AS STRING)), '[^0-9]', '')
   AND ndc_source.vocabulary_id = 'NDC'
   AND ndc_source.domain_id = 'Drug'
   AND ndc_source.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept_relationship ndc_maps_to
    ON ndc_maps_to.concept_id_1 = ndc_source.concept_id
   AND ndc_maps_to.relationship_id = 'Maps to'
   AND ndc_maps_to.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept ndc_standard
    ON ndc_standard.concept_id = ndc_maps_to.concept_id_2
   AND ndc_standard.standard_concept = 'S'
   AND ndc_standard.domain_id = 'Drug'
   AND ndc_standard.invalid_reason IS NULL
  WHERE ord.Active = TRUE
    AND ord.TypeCode = 'Medication'
    AND ord.ClientGUID IS NOT NULL
    AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
)
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN derived_drug_concept_id <> 0 THEN 1 ELSE 0 END) AS non_zero_drug_concept_id_rows,
  ROUND(100.0 * SUM(CASE WHEN derived_drug_concept_id <> 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS non_zero_fill_rate_pct,
  SUM(CASE WHEN winning_path IN ('rxnorm_standard', 'rxnorm_source') THEN 1 ELSE 0 END) AS rxnorm_won_rows,
  SUM(CASE WHEN winning_path IN ('ndc_standard', 'ndc_source') THEN 1 ELSE 0 END) AS ndc_won_rows,
  SUM(CASE WHEN winning_path = 'unmapped' THEN 1 ELSE 0 END) AS unmapped_rows
FROM mapping_candidates;